# YOLOv8x Fine-Tuning: Player and Ball Detection

**Purpose:** Fine-tune a single YOLOv8x model on the merged player/ball dataset and produce `models/players.pt`.  
**Inputs:** Roboflow dataset `basketball-player-ball-detection` v1; `yolov8x.pt` base weights; `augment_ball_images.py` (same folder).  
**Outputs:** the `runs/detect/players_train/` training run; its best checkpoint ships as `models/players.pt`.  
**Backs:** `results/training/detect_val/` and the tracking evaluation, where `players.pt` is the comparison configuration.  
**Environment:** QMUL JupyterHub (NVIDIA A40, ~46GB VRAM).

The model this notebook produced was later superseded as the pipeline's detector by a single seven-class checkpoint serving both player and ball detection; players.pt remains the comparison configuration in the tracking evaluation.

This notebook trains one model for both player and ball classes, rather than two separate models. This choice is justified by the scale of the merged training dataset (3,633 images): a single shared detection head is appropriate once sufficient data is available for both classes, and avoids the deployment overhead of maintaining two separate models in the downstream pipeline.

## Configuration

| Parameter | Value | Basis |
|---|---|---|
| Base model | `yolov8x.pt` | project-wide detector family |
| Dataset | Roboflow `basketball-player-ball-detection`, v1 | version pinned for reproducibility |
| Classes | `player`, `ball` | as labelled in the dataset |
| Epochs | 100 (ceiling) | early-stopping ceiling |
| Patience | 30 | arXiv:2408.05661 |
| Batch size | 32 | A40 ~46GB VRAM |
| Image size | 640x640 | Matches dataset preprocessing |
| Split | 80/10/10 | Roboflow dataset version, as-is |
| Output weight name | `players.pt` | training-time name; superseded in production |

## 1. Environment check

Confirms the GPU is visible and the conda environment is active before doing anything else. If this cell does not show CUDA as available, stop here and fix the environment first: training will silently fall back to CPU and take an unreasonable amount of time.

In [1]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Do not proceed with training until this is resolved.')

PyTorch version: 2.4.0+cu124
CUDA available: True
GPU: NVIDIA A40
VRAM: 47.8 GB


## 2. Install dependencies

Installs `ultralytics` (YOLOv8), `roboflow` (dataset download), and `albumentations` (targeted augmentation, Section 5). If these are already installed in your persistent JupyterHub environment, this cell is safe to re-run; pip will simply confirm they're already satisfied.

In [2]:
!pip install ultralytics roboflow albumentations --quiet

## 3. Download the dataset from Roboflow

This downloads the pinned dataset: workspace `hanad-ali`, project `basketball-player-ball-detection-fjmmg`, version 1, in YOLOv8 format.

In [3]:
import os

from roboflow import Roboflow

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace('hanad-ali').project('basketball-player-ball-detection-fjmmg')
dataset = project.version(1).download('yolov8')

print(f'Dataset downloaded to: {dataset.location}')

loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to: /home/jovyan/nba-video-analytics/training/basketball-player-ball-detection-1


## 4. Verify dataset structure and class balance

Before training, confirm the dataset is as expected: 2 classes (`player`, `ball`), and the expected annotation imbalance (player annotations far outnumber ball annotations). This is not a formality: training on an unexpected dataset version would invalidate the methodology section's reported figures.

In [4]:
import yaml
import os

data_yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print('Classes:', data_config['names'])
print('Number of classes:', data_config['nc'])
print()
print('Train path:', data_config['train'])
print('Val path:', data_config['val'])
print('Test path:', data_config.get('test', 'not specified'))

Classes: ['ball', 'player']
Number of classes: 2

Train path: ../train/images
Val path: ../valid/images
Test path: ../test/images


In [5]:
# Count annotations per class across the training split to confirm the
# documented imbalance (player: 15,634 / ball: 1,857 at dataset-version level).
# Re-counting here confirms THIS downloaded copy matches what was documented,
# rather than assuming it without checking.
from collections import Counter
import glob

train_labels_dir = os.path.join(dataset.location, 'train', 'labels')
label_files = glob.glob(os.path.join(train_labels_dir, '*.txt'))

class_counts = Counter()
for label_file in label_files:
    with open(label_file, 'r') as f:
        for line in f:
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

print(f'Label files scanned: {len(label_files)}')
for class_id, count in sorted(class_counts.items()):
    class_name = data_config['names'][class_id]
    print(f'Class {class_id} ({class_name}): {count} annotations')

Label files scanned: 2906
Class 0 (ball): 1489 annotations
Class 1 (player): 12501 annotations


## 5. Targeted augmentation for ball-labelled training images

The class imbalance confirmed above was acknowledged in advance as a limitation with a planned mitigation: if ball-class recall proved materially weaker than player-class recall after the first training run, targeted augmentation would be applied to ball-labelled images specifically, rather than augmenting the dataset uniformly. The first run measured a ball recall of 91.2% against a player recall of 97.4%, in line with what the imbalance predicted. This section runs that planned mitigation as a deliberate before/after experiment, to strengthen the evaluation evidence for the dissertation rather than as a corrective necessity.

The script below (`augment_ball_images.py`, in this same `training/` folder) filters the training split to images containing at least one ball annotation, and generates two augmented copies of each using a randomised pipeline of rotation, brightness/contrast jitter, motion blur, and scale jitter, each applied independently at random rather than always combined, under a fixed seed (42). Validation and test splits are left untouched, since augmenting evaluation data would invalidate the before/after comparison. Bounding boxes are updated correctly under the geometric transforms using Albumentations' YOLO-format bbox handling (Buslaev et al., 2020).

In [6]:
!python augment_ball_images.py --dataset-dir "{dataset.location}" --num-copies 2 --seed 42

Before augmentation: 1489 ball, 12501 player.
Found 1489 training images containing the ball class.
After augmentation: 4467 ball, 12501 player.
Ball:player ratio improved from 1:8.4 to 1:2.8.


In [7]:
# Re-measure class balance after augmentation, the same way the cell above
# measured it before, to confirm the augmentation step had the intended effect.
class_counts_after = Counter()
label_files_after = glob.glob(os.path.join(train_labels_dir, '*.txt'))
for label_file in label_files_after:
    with open(label_file, 'r') as f:
        for line in f:
            class_id = int(line.split()[0])
            class_counts_after[class_id] += 1

print(f'Label files after augmentation: {len(label_files_after)}')
for class_id, count in sorted(class_counts_after.items()):
    class_name = data_config['names'][class_id]
    print(f'Class {class_id} ({class_name}): {count} annotations')

Label files after augmentation: 5884
Class 0 (ball): 4467 annotations
Class 1 (player): 12501 annotations


## 6. Train the model

This is the only line required to fine-tune YOLOv8x on the augmented dataset. Parameters:

- `epochs=100`: maximum cap, not a fixed target
- `patience=30`: stop early if validation mAP does not improve for 30 consecutive epochs (matches arXiv:2408.05661)
- `batch=32`: sized to the A40's ~46GB VRAM
- `imgsz=640`: matches the dataset's preprocessing resize
- `plots=True`: generates training curves and confusion matrix for later inclusion in the dissertation
- training itself runs with Ultralytics' default `seed=0`

An explicit `project='runs/detect'` argument would duplicate Ultralytics' own default save-path behaviour for detection tasks (which already resolves to `runs/detect/<name>`), producing a doubled `runs/detect/runs/detect/players_train` path. `name='players_train'` alone is sufficient and produces the correct path.

Training time depends on dataset size and GPU load; on an A40 with this dataset, expect a long run. Confirm JupyterHub's keep-alive is active before starting it.

In [8]:
from ultralytics import YOLO

model = YOLO('yolov8x.pt')
results = model.train(
    data=os.path.join(dataset.location, 'data.yaml'),
    epochs=100,
    patience=30,
    batch=32,
    imgsz=640,
    plots=True,
    name='players_train'
)

New https://pypi.org/project/ultralytics/8.4.69 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/nba-video-analytics/training/basketball-player-ball-detection-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo

## 7. Per-class evaluation metrics

Precision, recall, and mAP are reported per class, not only as an aggregate figure, specifically because of the documented class imbalance. This cell extracts and prints those per-class figures explicitly so the imbalance's actual effect on ball detection is measured rather than assumed.

**What to do with this output:** record these per-class and overall metrics for inclusion in the dissertation methodology and results sections, and compare directly against the first run's figures (ball recall 91.2%, player recall 97.4%) to evaluate whether the targeted augmentation closed any of the gap.

In [9]:
# Validate the best checkpoint explicitly to get a clean metrics object
metrics = model.val()

print('=== Per-class results ===')
print(f"{'Class':<10} {'Precision':<12} {'Recall':<12} {'mAP50':<12} {'mAP50-95':<12}")
for i, class_name in enumerate(data_config['names']):
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    ap50 = metrics.box.ap50[i]
    ap = metrics.box.ap[i]
    print(f'{class_name:<10} {p:<12.4f} {r:<12.4f} {ap50:<12.4f} {ap:<12.4f}')

print()
print('=== Overall (all classes) ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
Model summary (fused): 113 layers, 68,125,494 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1546.2±328.4 MB/s, size: 78.7 KB)
val: Scanning /home/jovyan/nba-video-analytics/training/basketball-player-ball-detection-1/valid/labels.cache... 363 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 363/363 126.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.6s0.2s
                   all        363       1750      0.978      0.959      0.982      0.703
                  ball        182        182      0.983      0.945      0.971      0.631
                player        181       1568      0.974      0.973      0.992      0.774
Speed: 1.8ms preprocess, 16.2ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/jovyan/nba-video-analytics/training/runs/detect/val

## 8. Locate the best checkpoint

YOLOv8 saves two checkpoints during training: `best.pt` (best validation performance) and `last.pt` (final epoch). This project uses `best.pt`, renamed to `players.pt`. The cell below finds it automatically rather than relying on a hardcoded path that might not match the actual run folder name (Ultralytics appends a number if `players_train` already exists from a previous run, e.g. `players_train2`). This recursive search is kept as a safety net even though the path-doubling hazard is avoided at the source in Section 6.

In [10]:
import glob

# Find the most recently created best.pt anywhere under runs/
candidates = glob.glob('runs/**/players_train*/weights/best.pt', recursive=True)
candidates.sort(key=os.path.getmtime, reverse=True)

if not candidates:
    raise FileNotFoundError('No best.pt found. Check that training completed successfully.')

best_pt_path = candidates[0]
print(f'Found best checkpoint: {best_pt_path}')

Found best checkpoint: runs/detect/players_train/weights/best.pt


## 9. Outcome

The run produced `runs/detect/players_train/weights/best.pt`, shipped as `models/players.pt`: the tracking evaluation's comparison configuration. Production detection uses the seven-class checkpoint `models/ball.pt`.